# Supply Chain Disruption Agent

An agentic system that monitors supplier status, detects delivery delays,
discovers alternative suppliers, and simulates order rebalancing — driven by
OpenAI function calling with synthetic supplier data and the World Bank API
for country risk indicators.

## Why an Agent?

Supply chain disruptions are **cascading events**: a delayed shipment from one
supplier may force you to split orders, renegotiate lead times, and evaluate
country-level risks for alternative sources. A static pipeline cannot adapt
to each unique disruption scenario. An agent, by contrast, checks supplier
status, reasons about which products are at risk, searches for alternatives,
and simulates rebalancing — calling tools in whatever order the situation
demands.

In [ ]:
%pip install openai requests pandas --quiet

In [ ]:
import json, textwrap, random, time
import requests
import pandas as pd
from openai import OpenAI

client = OpenAI()
MODEL = "gpt-4o-mini"
random.seed(42)  # reproducible synthetic data

## Synthetic Supplier Data

We create a realistic supplier database that the agent tools will query.
Each supplier has products, lead times, capacity, country, and a current
status that may include delays.

In [ ]:
SUPPLIERS = {
    "SUP-001": {
        "name": "Shenzhen MicroElec",
        "country": "CHN",
        "products": ["microcontroller-A1", "sensor-B2"],
        "lead_time_days": 21,
        "capacity_units_month": 50000,
        "status": "delayed",
        "delay_days": 14,
        "delay_reason": "Port congestion at Yantian",
    },
    "SUP-002": {
        "name": "Taiwan SemiParts",
        "country": "TWN",
        "products": ["microcontroller-A1", "capacitor-C3"],
        "lead_time_days": 28,
        "capacity_units_month": 30000,
        "status": "operational",
        "delay_days": 0,
        "delay_reason": None,
    },
    "SUP-003": {
        "name": "Bavaria Precision GmbH",
        "country": "DEU",
        "products": ["sensor-B2", "housing-D4"],
        "lead_time_days": 14,
        "capacity_units_month": 20000,
        "status": "operational",
        "delay_days": 0,
        "delay_reason": None,
    },
    "SUP-004": {
        "name": "MexiParts SA",
        "country": "MEX",
        "products": ["housing-D4", "cable-E5"],
        "lead_time_days": 7,
        "capacity_units_month": 40000,
        "status": "at_risk",
        "delay_days": 3,
        "delay_reason": "Customs processing slowdown",
    },
    "SUP-005": {
        "name": "Bangalore TechSource",
        "country": "IND",
        "products": ["microcontroller-A1", "cable-E5", "sensor-B2"],
        "lead_time_days": 30,
        "capacity_units_month": 25000,
        "status": "operational",
        "delay_days": 0,
        "delay_reason": None,
    },
}

OPEN_ORDERS = [
    {"order_id": "ORD-100", "supplier_id": "SUP-001", "product": "microcontroller-A1", "quantity": 10000, "due_date": "2026-04-01"},
    {"order_id": "ORD-101", "supplier_id": "SUP-001", "product": "sensor-B2", "quantity": 5000, "due_date": "2026-04-05"},
    {"order_id": "ORD-102", "supplier_id": "SUP-004", "product": "housing-D4", "quantity": 8000, "due_date": "2026-03-25"},
    {"order_id": "ORD-103", "supplier_id": "SUP-003", "product": "housing-D4", "quantity": 3000, "due_date": "2026-03-28"},
]

print("=== Suppliers ===")
display(pd.DataFrame.from_dict(SUPPLIERS, orient="index")[["name", "country", "status", "delay_days", "products"]])
print("\n=== Open Orders ===")
display(pd.DataFrame(OPEN_ORDERS))

## Tool 1 — `get_supplier_status`

Returns the current status and details for a given supplier.

In [ ]:
def get_supplier_status(supplier_id: str) -> str:
    """Get the current status, delay info, and metadata for a supplier."""
    sup = SUPPLIERS.get(supplier_id.upper())
    if not sup:
        return json.dumps({"error": f"Supplier {supplier_id} not found"})

    # Include open orders for this supplier
    orders = [o for o in OPEN_ORDERS if o["supplier_id"] == supplier_id.upper()]
    return json.dumps({"supplier_id": supplier_id, **sup, "open_orders": orders}, indent=2)

## Tool 2 — `check_lead_times`

Returns all suppliers that carry a given product, with their lead times and
current capacity.

In [ ]:
def check_lead_times(product: str) -> str:
    """Check lead times and availability across all suppliers for a product."""
    results = []
    for sid, sup in SUPPLIERS.items():
        if product.lower() in [p.lower() for p in sup["products"]]:
            effective_lead = sup["lead_time_days"] + sup.get("delay_days", 0)
            results.append({
                "supplier_id": sid,
                "name": sup["name"],
                "country": sup["country"],
                "status": sup["status"],
                "base_lead_time_days": sup["lead_time_days"],
                "current_delay_days": sup.get("delay_days", 0),
                "effective_lead_time_days": effective_lead,
                "capacity_units_month": sup["capacity_units_month"],
            })
    results.sort(key=lambda x: x["effective_lead_time_days"])
    return json.dumps(results, indent=2)

## Tool 3 — `find_alternatives`

Finds alternative suppliers for a product, optionally filtered by region.
Includes a World Bank country risk indicator.

In [ ]:
def _get_country_risk(iso3: str) -> dict:
    """Fetch a logistics performance indicator from the World Bank API."""
    url = (
        f"https://api.worldbank.org/v2/country/{iso3}/indicator/"
        f"LP.LPI.OVRL.XQ?format=json&date=2023&per_page=1"
    )
    try:
        resp = requests.get(url, timeout=15)
        data = resp.json()
        if len(data) > 1 and data[1]:
            val = data[1][0].get("value")
            return {"lpi_score": round(val, 2) if val else None, "source": "World Bank LPI 2023"}
    except Exception:
        pass
    return {"lpi_score": None, "source": "unavailable"}


REGION_MAP = {
    "asia": ["CHN", "TWN", "IND", "JPN", "KOR"],
    "europe": ["DEU", "FRA", "GBR", "ITA", "POL"],
    "americas": ["USA", "MEX", "BRA", "CAN"],
}


def find_alternatives(product: str, region: str = "") -> str:
    """Find alternative suppliers for a product, optionally filtered by region
    ('asia', 'europe', 'americas'). Includes World Bank country risk."""
    region_codes = REGION_MAP.get(region.lower(), []) if region else []
    results = []
    for sid, sup in SUPPLIERS.items():
        if product.lower() not in [p.lower() for p in sup["products"]]:
            continue
        if region_codes and sup["country"] not in region_codes:
            continue
        risk = _get_country_risk(sup["country"])
        results.append({
            "supplier_id": sid,
            "name": sup["name"],
            "country": sup["country"],
            "status": sup["status"],
            "lead_time_days": sup["lead_time_days"],
            "capacity_units_month": sup["capacity_units_month"],
            "country_risk": risk,
        })
    return json.dumps(results, indent=2)

## Tool 4 — `calculate_rebalance`

Takes a list of at-risk orders and a list of alternative suppliers, then
simulates how to redistribute the orders to meet deadlines.

In [ ]:
from datetime import datetime, timedelta


def calculate_rebalance(orders: list[dict], alternatives: list[dict]) -> str:
    """Simulate rebalancing orders across alternative suppliers.

    Each order dict: {order_id, product, quantity, due_date}
    Each alternative dict: {supplier_id, name, lead_time_days, capacity_units_month}
    """
    today = datetime.now()
    plan = []

    for order in orders:
        due = datetime.strptime(order["due_date"], "%Y-%m-%d")
        days_left = (due - today).days
        qty_remaining = order["quantity"]
        allocations = []

        # Sort alternatives by lead time (fastest first)
        alts_sorted = sorted(alternatives, key=lambda a: a.get("lead_time_days", 999))

        for alt in alts_sorted:
            if qty_remaining <= 0:
                break
            if alt["lead_time_days"] > days_left:
                continue  # cannot meet deadline
            # Allocate up to monthly capacity
            available = alt.get("capacity_units_month", 0)
            alloc = min(qty_remaining, available)
            if alloc > 0:
                ship_date = today + timedelta(days=alt["lead_time_days"])
                allocations.append({
                    "supplier_id": alt["supplier_id"],
                    "name": alt["name"],
                    "allocated_qty": alloc,
                    "estimated_arrival": ship_date.strftime("%Y-%m-%d"),
                    "meets_deadline": ship_date <= due,
                })
                qty_remaining -= alloc

        plan.append({
            "order_id": order["order_id"],
            "product": order["product"],
            "original_quantity": order["quantity"],
            "unallocated_quantity": max(qty_remaining, 0),
            "fully_covered": qty_remaining <= 0,
            "allocations": allocations,
        })

    return json.dumps(plan, indent=2)

## Tool Schema for OpenAI Function Calling

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_supplier_status",
            "description": "Get current status, delay info, and open orders for a supplier.",
            "parameters": {
                "type": "object",
                "properties": {
                    "supplier_id": {"type": "string", "description": "Supplier ID, e.g. SUP-001"},
                },
                "required": ["supplier_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_lead_times",
            "description": "Check lead times and availability across all suppliers for a given product.",
            "parameters": {
                "type": "object",
                "properties": {
                    "product": {"type": "string", "description": "Product name, e.g. microcontroller-A1"},
                },
                "required": ["product"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "find_alternatives",
            "description": "Find alternative suppliers for a product, optionally filtered by region. Includes World Bank country risk data.",
            "parameters": {
                "type": "object",
                "properties": {
                    "product": {"type": "string", "description": "Product name"},
                    "region": {"type": "string", "enum": ["asia", "europe", "americas", ""], "description": "Optional region filter"},
                },
                "required": ["product"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_rebalance",
            "description": "Simulate rebalancing at-risk orders across alternative suppliers to meet deadlines.",
            "parameters": {
                "type": "object",
                "properties": {
                    "orders": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "order_id": {"type": "string"},
                                "product": {"type": "string"},
                                "quantity": {"type": "integer"},
                                "due_date": {"type": "string", "description": "YYYY-MM-DD"},
                            },
                        },
                    },
                    "alternatives": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "supplier_id": {"type": "string"},
                                "name": {"type": "string"},
                                "lead_time_days": {"type": "integer"},
                                "capacity_units_month": {"type": "integer"},
                            },
                        },
                    },
                },
                "required": ["orders", "alternatives"],
            },
        },
    }
]

## Dispatcher

In [ ]:
TOOL_MAP = {
    "get_supplier_status": get_supplier_status,
    "check_lead_times": check_lead_times,
    "find_alternatives": find_alternatives,
    "calculate_rebalance": calculate_rebalance,
}

## Agent Loop

In [ ]:
SYSTEM_PROMPT = textwrap.dedent("""\
    You are a Supply Chain Disruption analyst agent. Your job:
    1. Monitor supplier statuses and detect delays or risks.
    2. Check lead times for affected products.
    3. Find alternative suppliers (include country risk data).
    4. Simulate order rebalancing to see if deadlines can be met.
    5. Produce a disruption response plan with clear recommendations.

    Supplier IDs: SUP-001 through SUP-005.
    Products: microcontroller-A1, sensor-B2, capacitor-C3, housing-D4, cable-E5.

    Always check supplier status first, then investigate affected products.
    Use calculate_rebalance before writing the final plan.
""")


def run_agent(user_message: str, max_turns: int = 12) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]

    for turn in range(max_turns):
        print(f"\n--- Agent turn {turn + 1} ---")
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )
        msg = response.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            print("\n--- Agent finished ---")
            return msg.content

        for tc in msg.tool_calls:
            fn_name = tc.function.name
            fn_args = json.loads(tc.function.arguments)
            print(f"  Calling {fn_name}({json.dumps(fn_args)[:120]})")
            fn = TOOL_MAP.get(fn_name)
            result = fn(**fn_args) if fn else json.dumps({"error": f"Unknown tool: {fn_name}"})
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    return "Agent reached maximum turns without finishing."

## Example Execution

Investigate the disrupted supplier SUP-001 and determine a rebalancing plan.

In [ ]:
report = run_agent(
    "SUP-001 (Shenzhen MicroElec) appears to be delayed. Investigate the "
    "situation: check their status, see which products are affected, find "
    "alternatives (consider diversifying away from Asia), and simulate a "
    "rebalancing plan for the open orders. Give me a disruption response plan."
)

In [ ]:
from IPython.display import Markdown
Markdown(report)

## Second Scenario — Multi-Supplier Risk Scan

In [ ]:
report2 = run_agent(
    "Run a full supply chain risk scan. Check the status of all five suppliers "
    "(SUP-001 through SUP-005). For any that are delayed or at risk, check lead "
    "times for their products, find alternatives, and simulate rebalancing for "
    "affected open orders. Produce a comprehensive risk report."
)

In [ ]:
Markdown(report2)

## Results Analysis

| Aspect | Detail |
|--------|--------|
| **Data sources** | Synthetic supplier DB (in-notebook), World Bank LPI API (live) |
| **Adaptive flow** | Agent checked all suppliers, focused only on disrupted ones, and built a rebalancing plan |
| **Country risk** | World Bank Logistics Performance Index adds geopolitical context to alternative selection |
| **Limitations** | Rebalancing is a greedy simulation (not an optimizer); real procurement has contracts, MOQs, etc. |

### Potential Extensions

1. **Real ERP integration** — replace synthetic data with SAP/Oracle API calls.
2. **Cost optimization** — add shipping cost and tariff data to the rebalancing logic.
3. **Event monitoring** — add a tool that watches news/RSS feeds for disruption signals.
4. **Multi-tier visibility** — extend to sub-suppliers (tier 2+).

---
*Notebook by the LLM Course — Agentic Designs series*